In [ ]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
# ============================================================
# CELL 1
# plain_sentiment_df.csv OKU + train_df / val_df / test_df OLUŞTUR
#
# Dosya:
# D:\serkan.kaymak\financial_sentiment_thesis\app\data\plain_sentiment_df.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 0) Ayarlar
# ------------------------------------------------------------
PROJECT_DIR = PROJECT_ROOT

PLAIN_SENTIMENT_PATH = paths.PLAIN_SENTIMENT_DATASET_CSV_PATH

SPLIT_DIR = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "train_df.parquet"
VAL_PATH = SPLIT_DIR / "val_df.parquet"
TEST_PATH = SPLIT_DIR / "test_df.parquet"

RANDOM_STATE = 42

# İlk deneme için PhraseBank'i train'e ekleyelim mi?
# FinBERT PhraseBank'i görmüş olabilir; bu yüzden test'e ASLA koymuyoruz.
USE_PHRASEBANK_IN_TRAIN = True

# Eski splitleri aynen kullanmak istiyorsan True.
# Yeni split üretmek istiyorsan False yap.
REUSE_EXISTING_SPLITS = False

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

# ------------------------------------------------------------
# 1) Önceden kayıtlı split varsa oku
# ------------------------------------------------------------
if REUSE_EXISTING_SPLITS and TRAIN_PATH.exists() and VAL_PATH.exists() and TEST_PATH.exists():
    print("Mevcut split dosyaları bulundu. Okunuyor...")

    train_df = pd.read_parquet(TRAIN_PATH)
    val_df = pd.read_parquet(VAL_PATH)
    test_df = pd.read_parquet(TEST_PATH)

else:
    # ------------------------------------------------------------
    # 2) plain_sentiment_df.csv oku
    # ------------------------------------------------------------
    if not PLAIN_SENTIMENT_PATH.exists():
        raise FileNotFoundError(f"Dosya bulunamadı: {PLAIN_SENTIMENT_PATH}")

    print("Okunan dosya:", PLAIN_SENTIMENT_PATH)

    plain_sentiment_df = pd.read_csv(PLAIN_SENTIMENT_PATH)

    df = plain_sentiment_df.copy()

    print("\nRaw shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    # ------------------------------------------------------------
    # 3) Temizlik
    # ------------------------------------------------------------
    if "text" not in df.columns:
        raise ValueError("plain_sentiment_df içinde 'text' kolonu yok.")

    if "label" not in df.columns:
        raise ValueError("plain_sentiment_df içinde 'label' kolonu yok.")

    if "source_dataset" not in df.columns:
        raise ValueError("source_dataset kolonu yok. Twitter / PhraseBank ayrımı yapılamıyor.")

    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.lower().str.strip()

    df = df[
        (df["text"] != "") &
        (df["text"].str.lower() != "nan") &
        (df["label"].isin(["negative", "neutral", "positive"]))
    ].copy()

    df["text_norm"] = (
        df["text"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Aynı text + aynı label tekrarlarını temizle
    df = df.drop_duplicates(subset=["text_norm", "label"]).reset_index(drop=True)

    print("\nAfter cleaning:", df.shape)

    print("\nSource distribution:")
    print(df["source_dataset"].value_counts(dropna=False))

    print("\nLabel distribution:")
    print(df["label"].value_counts())

    # ------------------------------------------------------------
    # 4) Ana veri: TwitterFinancialNewsSentiment
    # ------------------------------------------------------------
    twitter_df = df[df["source_dataset"] == "TwitterFinancialNewsSentiment"].copy()

    if len(twitter_df) == 0:
        raise ValueError("TwitterFinancialNewsSentiment satırı bulunamadı.")

    # Eğer uygunluk kolonları varsa kullan
    if "usable_for_main_evaluation" in twitter_df.columns:
        twitter_df = twitter_df[twitter_df["usable_for_main_evaluation"] == True].copy()

    if "finbert_comparison_suitable" in twitter_df.columns:
        twitter_df = twitter_df[twitter_df["finbert_comparison_suitable"] == True].copy()

    twitter_df = twitter_df.reset_index(drop=True)

    print("\nTwitter data:", twitter_df.shape)
    print(twitter_df["label"].value_counts())

    # ------------------------------------------------------------
    # 5) Twitter'dan train / val / test split
    # Test tamamen Twitter'dan gelecek.
    # PhraseBank test'e girmeyecek.
    # ------------------------------------------------------------
    twitter_train_val_df, test_df = train_test_split(
        twitter_df,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=twitter_df["label"]
    )

    twitter_train_df, val_df = train_test_split(
        twitter_train_val_df,
        test_size=0.15,
        random_state=RANDOM_STATE,
        stratify=twitter_train_val_df["label"]
    )

    train_parts = [twitter_train_df.copy()]

    # ------------------------------------------------------------
    # 6) Financial PhraseBank sadece train'e eklenebilir
    # ------------------------------------------------------------
    if USE_PHRASEBANK_IN_TRAIN:
        phrasebank_df = df[df["source_dataset"] == "FinancialPhraseBank"].copy()

        if len(phrasebank_df) > 0:
            test_texts = set(test_df["text_norm"])
            phrasebank_df = phrasebank_df[~phrasebank_df["text_norm"].isin(test_texts)].copy()

            train_parts.append(phrasebank_df)

            print("\nFinancialPhraseBank train'e eklendi:", phrasebank_df.shape)
            print(phrasebank_df["label"].value_counts())
        else:
            print("\nFinancialPhraseBank bulunamadı, train'e eklenmedi.")

    train_df = pd.concat(train_parts, ignore_index=True)

    # ------------------------------------------------------------
    # 7) input_text ve label_id oluştur
    # ------------------------------------------------------------
    final_splits = {}

    for split_name, split_df in {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }.items():
        temp = split_df.copy()

        temp["input_text"] = temp["text"].astype(str).str.strip()
        temp["label"] = temp["label"].astype(str).str.lower().str.strip()
        temp["label_id"] = temp["label"].map(LABEL2ID)

        temp = temp[
            (temp["input_text"] != "") &
            (temp["label"].isin(LABEL2ID.keys())) &
            (temp["label_id"].notna())
        ].copy()

        temp["label_id"] = temp["label_id"].astype(int)

        temp = temp.reset_index(drop=True)

        final_splits[split_name] = temp

    train_df = final_splits["train"]
    val_df = final_splits["val"]
    test_df = final_splits["test"]

    # ------------------------------------------------------------
    # 8) Leakage kontrol
    # ------------------------------------------------------------
    train_texts = set(train_df["input_text"].astype(str).str.lower().str.strip())
    val_texts = set(val_df["input_text"].astype(str).str.lower().str.strip())
    test_texts = set(test_df["input_text"].astype(str).str.lower().str.strip())

    train_val_overlap = len(train_texts.intersection(val_texts))
    train_test_overlap = len(train_texts.intersection(test_texts))
    val_test_overlap = len(val_texts.intersection(test_texts))

    print("\nLeakage check:")
    print("train-val overlap :", train_val_overlap)
    print("train-test overlap:", train_test_overlap)
    print("val-test overlap  :", val_test_overlap)

    if train_test_overlap > 0:
        raise ValueError("Train-test text overlap var. Test sızıntısı olabilir.")

    # ------------------------------------------------------------
    # 9) Splitleri kaydet
    # ------------------------------------------------------------
    train_df.to_parquet(TRAIN_PATH, index=False)
    val_df.to_parquet(VAL_PATH, index=False)
    test_df.to_parquet(TEST_PATH, index=False)

    print("\nSplitler kaydedildi:")
    print("Train:", TRAIN_PATH)
    print("Val  :", VAL_PATH)
    print("Test :", TEST_PATH)

# ------------------------------------------------------------
# 10) Final özet
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("SPLITS READY")
print("=" * 100)

for name, split_df in {
    "train_df": train_df,
    "val_df": val_df,
    "test_df": test_df,
}.items():
    print("\n" + name, split_df.shape)

    print("\nLabel count:")
    print(split_df["label"].value_counts())

    print("\nLabel ratio:")
    print(split_df["label"].value_counts(normalize=True).mul(100).round(2))

    if "source_dataset" in split_df.columns:
        print("\nSource count:")
        print(split_df["source_dataset"].value_counts(dropna=False))

print("\nTrain sample:")
display(train_df[["input_text", "label", "label_id"]].head(PREVIEW_ROWS))

print("\nVal sample:")
display(val_df[["input_text", "label", "label_id"]].head(PREVIEW_ROWS))

print("\nTest sample:")
display(test_df[["input_text", "label", "label_id"]].head(PREVIEW_ROWS))

In [ ]:
# ============================================================
# CELL 2
# MULTI-MODEL FINETUNE
# CHECKPOINT AÇIK
# YARIDA KESİLİRSE KALDIĞI YERDEN DEVAM EDER
#
# Gerekli hazır değişkenler:
# train_df, val_df, test_df
#
# Her model için:
# checkpoints root:
# D:\serkan.kaymak\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\<run_name>\checkpoints
#
# final model:
# D:\serkan.kaymak\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\<run_name>\final_model
# ============================================================

from pathlib import Path
import os
import sys
import json
import gc
import inspect
import random
import subprocess

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from transformers.trainer_utils import get_last_checkpoint

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# 0) Paket kontrol
# ------------------------------------------------------------
def pip_install(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

pip_install("datasets")
pip_install("transformers")
pip_install("torch")
pip_install("sklearn", "scikit-learn")
pip_install("accelerate")


# ------------------------------------------------------------
# 1) Genel ayarlar
# ------------------------------------------------------------
PROJECT_DIR = PROJECT_ROOT

CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = paths.MODEL_RESULTS_ROOT / "training_evaluation"

TEXT_COL = "input_text"
LABEL_COL = "label"
LABEL_ID_COL = "label_id"

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

NUM_LABELS = 3

RANDOM_STATE = 42

MAX_LENGTH = 128
EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 3

USE_CLASS_WEIGHTS = True

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Checkpoint root:")
print(CHECKPOINT_ROOT)


# ------------------------------------------------------------
# 2) Modeller
# ------------------------------------------------------------
MODEL_RUNS = [
    {
        "run_name": "bert_base_uncased",
        "model_name": "bert-base-uncased",
    },
    {
        "run_name": "distilbert_base_uncased",
        "model_name": "distilbert-base-uncased",
    },
    {
        "run_name": "roberta_base",
        "model_name": "roberta-base",
    },
]

# İstersen sonradan şunu da ekleyebiliriz:
# {
#     "run_name": "albert_base_v2",
#     "model_name": "albert-base-v2",
# },


# ------------------------------------------------------------
# 3) Seed
# ------------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)


# ------------------------------------------------------------
# 4) Veri kontrol
# ------------------------------------------------------------
for df_name in ["train_df", "val_df", "test_df"]:
    if df_name not in globals():
        raise ValueError(f"{df_name} bulunamadı. Önce CELL 1 split hücresini çalıştır.")

for name, split_df in {
    "train_df": train_df,
    "val_df": val_df,
    "test_df": test_df,
}.items():
    for col in [TEXT_COL, LABEL_COL, LABEL_ID_COL]:
        if col not in split_df.columns:
            raise ValueError(f"{name} içinde eksik kolon: {col}")

    print(name, split_df.shape)
    print(split_df[LABEL_COL].value_counts())
    print()


# ------------------------------------------------------------
# 5) Dataset hazırlama fonksiyonu
# ------------------------------------------------------------
def prepare_hf_dataset(split_df, tokenizer, max_length=128):
    temp = split_df[[TEXT_COL, LABEL_ID_COL]].copy()
    temp[TEXT_COL] = temp[TEXT_COL].astype(str)
    temp[LABEL_ID_COL] = temp[LABEL_ID_COL].astype(int)

    temp = temp.rename(columns={LABEL_ID_COL: "labels"})

    ds = Dataset.from_pandas(temp.reset_index(drop=True))

    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    ds = ds.map(tokenize_fn, batched=True)

    keep_cols = ["input_ids", "attention_mask", "labels"]

    if "token_type_ids" in ds.column_names:
        keep_cols.append("token_type_ids")

    ds.set_format(type="torch", columns=keep_cols)

    return ds


# ------------------------------------------------------------
# 6) Trainer tokenizer uyumluluğu
# ------------------------------------------------------------
def get_trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)

    if "processing_class" in sig.parameters:
        return {"processing_class": tokenizer}

    if "tokenizer" in sig.parameters:
        return {"tokenizer": tokenizer}

    return {}


# ------------------------------------------------------------
# 7) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def evaluate_model(trainer, dataset, split_name, run_name):
    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    y_true = pred_output.label_ids
    y_pred = np.argmax(logits, axis=-1)

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    labels_order = [0, 1, 2]
    target_names = [ID2LABEL[i] for i in labels_order]

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels_order,
        target_names=target_names,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels_order
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{ID2LABEL[i]}" for i in labels_order],
        columns=[f"pred_{ID2LABEL[i]}" for i in labels_order],
    )

    metrics = {
        "run_name": run_name,
        "split": split_name,
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "n_eval": int(len(y_true)),
    }

    print("\n" + "=" * 100)
    print(f"{run_name} | {split_name.upper()} RESULT")
    print("=" * 100)
    print(f"Accuracy   : {acc:.4f}")
    print(f"F1 Macro   : {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")
    print("\nClassification Report:")
    print(report_text)
    print("\nConfusion Matrix:")
    display(cm_df)

    return metrics


# ------------------------------------------------------------
# 8) TrainingArguments
# ------------------------------------------------------------
def build_training_args(output_dir):
    sig = inspect.signature(TrainingArguments.__init__)

    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
    )

    # eval strategy
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "epoch"

    # save strategy
    if "save_strategy" in sig.parameters:
        kwargs["save_strategy"] = "epoch"

    if "save_total_limit" in sig.parameters:
        kwargs["save_total_limit"] = SAVE_TOTAL_LIMIT

    if "load_best_model_at_end" in sig.parameters:
        kwargs["load_best_model_at_end"] = True

    if "metric_for_best_model" in sig.parameters:
        kwargs["metric_for_best_model"] = "f1_macro"

    if "greater_is_better" in sig.parameters:
        kwargs["greater_is_better"] = True

    if "logging_strategy" in sig.parameters:
        kwargs["logging_strategy"] = "steps"

    if "fp16" in sig.parameters:
        kwargs["fp16"] = torch.cuda.is_available()

    return TrainingArguments(**kwargs)


# ------------------------------------------------------------
# 9) Weighted Trainer
# ------------------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights_tensor is not None:
            weights = self.class_weights_tensor.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


# ------------------------------------------------------------
# 10) Tek modeli eğitme fonksiyonu
# ------------------------------------------------------------
def train_one_model(run_cfg):
    run_name = run_cfg["run_name"]
    model_name = run_cfg["model_name"]

    print("\n" + "#" * 120)
    print(f"MODEL BAŞLIYOR: {run_name} | {model_name}")
    print("#" * 120)

    run_dir = CHECKPOINT_ROOT / run_name
    checkpoint_dir = run_dir / "checkpoints"
    final_model_dir = run_dir / "final_model"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    final_model_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Tokenizer / dataset
    # --------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    train_ds = prepare_hf_dataset(train_df, tokenizer, max_length=MAX_LENGTH)
    val_ds = prepare_hf_dataset(val_df, tokenizer, max_length=MAX_LENGTH)
    test_ds = prepare_hf_dataset(test_df, tokenizer, max_length=MAX_LENGTH)

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    model.to(device)

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------
    class_weights_tensor = None

    if USE_CLASS_WEIGHTS:
        train_labels = train_df[LABEL_ID_COL].astype(int).values
        counts = np.bincount(train_labels, minlength=NUM_LABELS)

        class_weights = []
        for class_id in range(NUM_LABELS):
            if counts[class_id] == 0:
                class_weights.append(1.0)
            else:
                class_weights.append(len(train_labels) / (NUM_LABELS * counts[class_id]))

        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

        print("\nClass counts:", counts)
        print("Class weights:")
        for i, w in enumerate(class_weights):
            print(i, ID2LABEL[i], "->", round(float(w), 4))

    # --------------------------------------------------------
    # Trainer
    # --------------------------------------------------------
    training_args = build_training_args(checkpoint_dir)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights_tensor=class_weights_tensor,
        **get_trainer_tokenizer_kwargs(tokenizer),
    )

    # --------------------------------------------------------
    # Resume checkpoint
    # --------------------------------------------------------
    last_checkpoint = None

    if checkpoint_dir.exists():
        last_checkpoint = get_last_checkpoint(str(checkpoint_dir))

    if last_checkpoint is not None:
        print("\nCheckpoint bulundu. Eğitim buradan devam edecek:")
        print(last_checkpoint)
        train_output = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("\nCheckpoint bulunamadı. Eğitim sıfırdan başlayacak.")
        train_output = trainer.train()

    print("\nTraining output:")
    print(train_output)

    # --------------------------------------------------------
    # Final model kaydet
    # --------------------------------------------------------
    print("\nFinal model kaydediliyor:")
    print(final_model_dir)

    trainer.save_model(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------
    val_metrics = evaluate_model(
        trainer=trainer,
        dataset=val_ds,
        split_name="val",
        run_name=run_name,
    )

    test_metrics = evaluate_model(
        trainer=trainer,
        dataset=test_ds,
        split_name="test",
        run_name=run_name,
    )

    # --------------------------------------------------------
    # Run config kaydet
    # --------------------------------------------------------
    run_config = {
        "run_name": run_name,
        "model_name": model_name,
        "max_length": MAX_LENGTH,
        "epochs": EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "checkpoint_dir": str(checkpoint_dir),
        "final_model_dir": str(final_model_dir),
    }

    with open(run_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(run_config, f, ensure_ascii=False, indent=2)

    # --------------------------------------------------------
    # Bellek temizliği
    # --------------------------------------------------------
    del model
    del tokenizer
    del trainer
    del train_ds
    del val_ds
    del test_ds

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run_name": run_name,
        "model_name": model_name,
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "test_accuracy": test_metrics["accuracy"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        "checkpoint_dir": str(checkpoint_dir),
        "final_model_dir": str(final_model_dir),
    }


# ------------------------------------------------------------
# 11) Tüm modelleri sırayla eğit
# ------------------------------------------------------------
all_results = []

for run_cfg in MODEL_RUNS:
    result = train_one_model(run_cfg)
    all_results.append(result)


summary_df = pd.DataFrame(all_results)

summary_df = summary_df.sort_values(
    "test_f1_macro",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 100)
print("MULTI MODEL TRAINING FINISHED")
print("=" * 100)

display(summary_df)

print("\nCheckpoint root:")
print(CHECKPOINT_ROOT)

# 04 - Plain Sentiment Modellerinin Fine-Tuning Deneyi

## 1. Deneyin Amacı

Bu deneyde finansal kısa metinlerde piyasa yönelimli duygu sınıflandırması problemi ele alınmıştır. Amaç, hazır bir finansal sentiment modeli olan FinBERT ile hedef veri seti üzerinde fine-tune edilen genel amaçlı Transformer modellerini karşılaştırmaktır.

Bu kapsamda üç farklı Transformer modeli fine-tune edilmiştir:

- BERT-base-uncased
- DistilBERT-base-uncased
- RoBERTa-base

Hazır FinBERT modeli ise baseline olarak değerlendirilmiştir.

---

## 2. Veri Seti Dağılımı

Deneyde kullanılan veri seti üçe ayrılmıştır:

| Split | Toplam Örnek | Negative | Neutral | Positive |
|---|---:|---:|---:|---:|
| Train | 12.950 | 1.820 | 8.137 | 2.993 |
| Validation | 1.432 | 215 | 929 | 288 |
| Test | 2.386 | 358 | 1.549 | 479 |

Veri dağılımı incelendiğinde `neutral` sınıfının belirgin biçimde baskın olduğu görülmektedir. Bu nedenle yalnızca accuracy metriğiyle değerlendirme yapmak yeterli değildir. Bu çalışmada ana değerlendirme metriği olarak **macro-F1** kullanılmıştır.

Macro-F1, her sınıfın F1 skorunu eşit ağırlıkla hesaba kattığı için sınıf dengesizliği bulunan veri setlerinde daha sağlıklı bir değerlendirme sunmaktadır.

---

## 3. Sınıf Dengesizliği ve Class Weight Kullanımı

Eğitim verisindeki sınıf dağılımı şu şekildedir:

| Sınıf | Örnek Sayısı |
|---|---:|
| Negative | 1.820 |
| Neutral | 8.137 |
| Positive | 2.993 |

Bu dağılım nedeniyle modelin çoğunluk sınıfı olan `neutral` sınıfına aşırı yönelmesini engellemek için class weight kullanılmıştır.

Kullanılan class weight değerleri:

| Sınıf | Class Weight |
|---|---:|
| Negative | 2.3718 |
| Neutral | 0.5305 |
| Positive | 1.4423 |

Bu yaklaşım, özellikle azınlık sınıfları olan `negative` ve `positive` sınıflarının model tarafından daha iyi öğrenilmesine yardımcı olmuştur.

---

## 4. Hazır FinBERT Baseline Sonucu

Daha önce hazır `ProsusAI/finbert` modeli test veri setinde değerlendirilmiştir.

| Model | Test Accuracy | Test Macro-F1 | Test Weighted-F1 |
|---|---:|---:|---:|
| Original FinBERT | 0.7323 | 0.6794 | 0.7400 |

Bu sonuç, hazır FinBERT modelinin finansal sentiment konusunda önceden eğitilmiş olmasına rağmen, hedef veri setindeki piyasa yönelimli etiket mantığına tam olarak uyum sağlayamadığını göstermektedir.

---

## 5. Fine-Tune Edilen Modellerin Genel Sonuçları

Fine-tune edilen modellerin validation ve test sonuçları aşağıdaki gibidir:

| Sıra | Model | Validation Accuracy | Validation Macro-F1 | Validation Weighted-F1 | Test Accuracy | Test Macro-F1 | Test Weighted-F1 |
|---:|---|---:|---:|---:|---:|---:|---:|
| 1 | RoBERTa-base | 0.8939 | 0.8718 | 0.8948 | 0.8906 | 0.8651 | 0.8918 |
| 2 | BERT-base-uncased | 0.8820 | 0.8535 | 0.8833 | 0.8722 | 0.8378 | 0.8733 |
| 3 | DistilBERT-base-uncased | 0.8792 | 0.8426 | 0.8801 | 0.8583 | 0.8218 | 0.8593 |
| 4 | Original FinBERT | - | - | - | 0.7323 | 0.6794 | 0.7400 |

Sonuçlara göre en iyi performans **RoBERTa-base** modelinde elde edilmiştir.

---

## 6. FinBERT ile Karşılaştırmalı Performans Artışı

Hazır FinBERT baseline ile fine-tune edilen modeller arasındaki farklar aşağıdaki gibidir:

| Model | Accuracy Artışı | Macro-F1 Artışı | Weighted-F1 Artışı |
|---|---:|---:|---:|
| DistilBERT-base-uncased | +0.1260 | +0.1424 | +0.1193 |
| BERT-base-uncased | +0.1399 | +0.1584 | +0.1333 |
| RoBERTa-base | +0.1583 | +0.1857 | +0.1518 |

Bu sonuçlar, hedef veri seti üzerinde yapılan fine-tuning işleminin hazır FinBERT kullanımına kıyasla belirgin bir performans artışı sağladığını göstermektedir.

Özellikle RoBERTa-base modeli, hazır FinBERT’e göre test macro-F1 değerinde yaklaşık **0.186 puanlık** bir artış sağlamıştır.

---

## 7. Model Bazlı Detaylı Analiz

## 7.1. BERT-base-uncased

### Validation Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8820 |
| Macro-F1 | 0.8535 |
| Weighted-F1 | 0.8833 |

### Test Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8722 |
| Macro-F1 | 0.8378 |
| Weighted-F1 | 0.8733 |

### Test Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.7429 | 0.7989 | 0.7699 | 358 |
| Neutral | 0.9234 | 0.8954 | 0.9092 | 1549 |
| Positive | 0.8176 | 0.8518 | 0.8344 | 479 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 286 | 59 | 13 |
| True Neutral | 84 | 1387 | 78 |
| True Positive | 15 | 56 | 408 |

BERT-base modeli test setinde güçlü bir performans göstermiştir. Özellikle `neutral` ve `positive` sınıflarında yüksek F1 değerleri elde edilmiştir. Ancak `negative` sınıfında RoBERTa’ya göre daha zayıf kalmıştır.

---

## 7.2. DistilBERT-base-uncased

### Validation Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8792 |
| Macro-F1 | 0.8426 |
| Weighted-F1 | 0.8801 |

### Test Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8583 |
| Macro-F1 | 0.8218 |
| Weighted-F1 | 0.8593 |

### Test Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.7480 | 0.7793 | 0.7633 | 358 |
| Neutral | 0.9117 | 0.8864 | 0.8989 | 1549 |
| Positive | 0.7811 | 0.8267 | 0.8032 | 479 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 279 | 62 | 17 |
| True Neutral | 82 | 1373 | 94 |
| True Positive | 12 | 71 | 396 |

DistilBERT, diğer modellere göre daha hafif ve daha hızlı bir mimaridir. Eğitim süresi BERT-base ve RoBERTa’ya göre daha kısadır. Buna rağmen hazır FinBERT baseline’ını açık biçimde geçmiştir. Ancak fine-tune edilen üç model arasında en düşük test macro-F1 değeri DistilBERT’te elde edilmiştir.

Bu nedenle DistilBERT, daha hızlı ve daha düşük maliyetli bir alternatif olarak değerlendirilebilir; fakat en yüksek başarı için RoBERTa veya BERT-base daha uygundur.

---

## 7.3. RoBERTa-base

### Validation Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8939 |
| Macro-F1 | 0.8718 |
| Weighted-F1 | 0.8948 |

### Test Sonucu

| Metrik | Değer |
|---|---:|
| Accuracy | 0.8906 |
| Macro-F1 | 0.8651 |
| Weighted-F1 | 0.8918 |

### Test Sınıf Bazlı Sonuçlar

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.8047 | 0.8631 | 0.8329 | 358 |
| Neutral | 0.9432 | 0.8999 | 0.9210 | 1549 |
| Positive | 0.8053 | 0.8810 | 0.8415 | 479 |

### Confusion Matrix

| Gerçek / Tahmin | Pred Negative | Pred Neutral | Pred Positive |
|---|---:|---:|---:|
| True Negative | 309 | 35 | 14 |
| True Neutral | 67 | 1394 | 88 |
| True Positive | 8 | 49 | 422 |

RoBERTa-base modeli hem validation hem de test setinde en yüksek performansı elde etmiştir. Test setinde 0.8906 accuracy, 0.8651 macro-F1 ve 0.8918 weighted-F1 değerlerine ulaşmıştır.

RoBERTa’nın en dikkat çekici yönü, yalnızca `neutral` sınıfında değil, azınlık sınıfları olan `negative` ve `positive` sınıflarında da güçlü performans göstermesidir.

Özellikle:

- Negative recall: 0.8631
- Positive recall: 0.8810
- Neutral F1-score: 0.9210

Bu sonuçlar, RoBERTa’nın sınıflar arasında daha dengeli bir sınıflandırma yaptığını göstermektedir.

---

## 8. Modellerin Sınıf Bazında Karşılaştırılması

### Test F1-score Karşılaştırması

| Model | Negative F1 | Neutral F1 | Positive F1 | Macro-F1 |
|---|---:|---:|---:|---:|
| DistilBERT-base-uncased | 0.7633 | 0.8989 | 0.8032 | 0.8218 |
| BERT-base-uncased | 0.7699 | 0.9092 | 0.8344 | 0.8378 |
| RoBERTa-base | 0.8329 | 0.9210 | 0.8415 | 0.8651 |

RoBERTa-base modeli her üç sınıfta da en dengeli performansı göstermiştir. Özellikle `negative` sınıfındaki F1-score değeri BERT-base ve DistilBERT’e göre belirgin biçimde daha yüksektir.

---

## 9. Overfitting ve Validation Loss Analizi

RoBERTa modelinin epoch bazlı validation sonuçları aşağıdaki gibidir:

| Epoch | Training Loss | Validation Loss | Accuracy | Macro-F1 | Weighted-F1 |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.4546 | 0.3794 | 0.8366 | 0.8089 | 0.8429 |
| 2 | 0.2783 | 0.3357 | 0.8862 | 0.8650 | 0.8879 |
| 3 | 0.2522 | 0.4148 | 0.8897 | 0.8660 | 0.8912 |
| 4 | 0.1393 | 0.5622 | 0.8939 | 0.8718 | 0.8948 |

RoBERTa modelinde validation loss değeri ikinci epoch’tan sonra yükselmiştir. Bu durum hafif overfitting veya model kalibrasyonunda bozulma işareti olarak değerlendirilebilir. Ancak macro-F1 ve accuracy değerleri dördüncü epoch’a kadar artmaya devam etmiştir.

Bu nedenle model seçiminde validation loss yerine macro-F1 metriği esas alınmıştır. Bunun temel nedeni, veri setinde sınıf dengesizliği bulunmasıdır. Macro-F1, negative, neutral ve positive sınıflarını eşit ağırlıkla değerlendirdiği için bu problemde validation loss’a kıyasla daha uygun bir seçim metriğidir.

Ayrıca RoBERTa modelinin test setinde de en yüksek performansı göstermesi, validation macro-F1’e göre yapılan model seçiminin genelleme açısından da başarılı olduğunu göstermektedir.

---

## 10. Neden Validation Loss Yerine Macro-F1 Kullanıldı?

Bu çalışmada validation loss yerine macro-F1 metriğinin model seçiminde esas alınmasının temel nedenleri şunlardır:

1. Veri seti sınıf dengesizliği içermektedir.
2. `neutral` sınıfı veri setinde baskın durumdadır.
3. Accuracy ve loss değerleri çoğunluk sınıfındaki başarıdan fazla etkilenebilir.
4. Macro-F1 her sınıfı eşit ağırlıkla değerlendirir.
5. Finansal sentiment probleminde azınlık sınıfları olan `negative` ve `positive` sınıflarındaki başarı da en az `neutral` kadar önemlidir.

Validation loss modelin olasılık güvenine duyarlıdır. Bu nedenle bazı durumlarda model daha fazla doğru tahmin yaparken, yanlış tahminlerinde daha yüksek güven ürettiği için validation loss artabilir. Bu çalışmada da bazı modellerde validation loss artışı gözlenmiş, ancak macro-F1 değeri artmaya devam etmiştir.

Bu nedenle model seçiminde ana görev metriği olarak macro-F1 kullanılmıştır.

---

## 11. Genel Model Sıralaması

Deney sonuçlarına göre model sıralaması şu şekildedir:

| Sıra | Model | Genel Değerlendirme |
|---:|---|---|
| 1 | RoBERTa-base | En iyi genel performans, en yüksek test macro-F1 |
| 2 | BERT-base-uncased | Güçlü performans, RoBERTa’nın gerisinde |
| 3 | DistilBERT-base-uncased | Daha hızlı ve hafif, ancak performansı daha düşük |
| 4 | Original FinBERT | Fine-tune edilen modellere göre belirgin biçimde düşük |

Genel sıralama:

```text
RoBERTa-base > BERT-base-uncased > DistilBERT-base-uncased > Original FinBERT


## 12. Ana Bulgular

Bu deneylerden elde edilen temel bulgular şunlardır:

1. Hedef veri seti üzerinde fine-tune edilen tüm Transformer modelleri hazır FinBERT baseline’ını geçmiştir.
2. En iyi sonuç RoBERTa-base modeliyle elde edilmiştir.
3. RoBERTa-base test setinde 0.8906 accuracy ve 0.8651 macro-F1 elde etmiştir.
4. BERT-base-uncased modeli güçlü bir ikinci model olarak öne çıkmıştır.
5. DistilBERT-base-uncased daha hızlı ve hafif olmasına rağmen performans açısından diğer iki modelin gerisinde kalmıştır.
6. Hazır FinBERT’in düşük performansı, hedef veri setinin etiket yapısına özel fine-tuning ihtiyacını göstermektedir.
7. Sınıf dengesizliği nedeniyle macro-F1 metriği model karşılaştırması için en uygun ana metrik olarak değerlendirilmiştir.

---

## 13. Tez İçin Yorum

Bu sonuçlar, finansal sentiment analizinde yalnızca finansal alanda önceden eğitilmiş bir model kullanmanın yeterli olmayabileceğini göstermektedir. Hazır FinBERT modeli finansal metinler üzerinde geliştirilmiş olsa da, hedef veri setindeki piyasa yönelimli sentiment etiketlerini doğrudan yüksek başarıyla sınıflandıramamıştır.

Buna karşılık, genel amaçlı Transformer modelleri hedef veri seti üzerinde fine-tune edildiğinde çok daha yüksek performans elde edilmiştir. Bu durum, finansal sentiment problemlerinde hedef veri setinin etiket yapısına uygun fine-tuning işleminin kritik olduğunu göstermektedir.

Özellikle RoBERTa-base modeli, hem validation hem de test setinde en yüksek performansı elde etmiş ve bu çalışmadaki en başarılı model olarak belirlenmiştir.

---

## 14. Tez Sonuç Paragrafı

Deneysel sonuçlara göre, hedef veri seti üzerinde fine-tune edilen Transformer tabanlı modeller hazır FinBERT baseline’ından belirgin biçimde daha yüksek performans göstermiştir. Hazır FinBERT modeli test setinde 0.7323 accuracy ve 0.6794 macro-F1 elde ederken, fine-tune edilen RoBERTa-base modeli 0.8906 accuracy ve 0.8651 macro-F1 değerlerine ulaşmıştır. BERT-base-uncased modeli 0.8722 accuracy ve 0.8378 macro-F1, DistilBERT-base-uncased modeli ise 0.8583 accuracy ve 0.8218 macro-F1 elde etmiştir.

Bu sonuçlar, hedef veri setinin piyasa yönelimli sentiment etiket yapısına özel fine-tuning işleminin, hazır finansal sentiment modeli kullanımına kıyasla önemli performans artışı sağladığını göstermektedir.

---



## 15. Sonuç

Bu aşamadaki deneyler sonucunda tez için güçlü bir model karşılaştırma zemini oluşmuştur. Fine-tune edilen modellerin hazır FinBERT baseline’ını belirgin biçimde geçmesi, çalışmanın temel hipotezini desteklemektedir.

Bu aşamada en iyi model:

**RoBERTa-base**

Ana sonuç:

**Hedef veri setine özel fine-tuning, hazır FinBERT kullanımına göre belirgin performans artışı sağlamıştır.**

##  Hocaya Sunulabilecek Kısa Özet

Hocam, üç farklı Transformer modelini fine-tune ettim: BERT-base, DistilBERT ve RoBERTa. Daha önce hazır FinBERT baseline test setinde 0.732 accuracy ve 0.679 macro-F1 elde etmişti. Fine-tune edilen modellerin tamamı FinBERT’i geçti. DistilBERT 0.858 accuracy ve 0.822 macro-F1, BERT-base 0.872 accuracy ve 0.838 macro-F1, RoBERTa ise 0.891 accuracy ve 0.865 macro-F1 elde etti.

En iyi sonuç RoBERTa-base modelinde alındı. RoBERTa ayrıca negative, neutral ve positive sınıflarında daha dengeli performans gösterdi. Validation loss bazı epoch’larda yükselse de macro-F1 artmaya devam ettiği ve test setinde de en yüksek sonuç RoBERTa’da elde edildiği için final model olarak RoBERTa-base daha uygun görünüyor.

Bundan sonraki aşamada bu modelleri ayrıca bağımsız olarak etiketlediğimiz S&P 500 headline test setinde de değerlendireceğiz.

---